In [1]:
import os
import sys

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
from datasets import Dataset

from utils.helpers import get_system_prompt, get_task_type, get_default_response, make_prompt, parse_output, make_demonstrations, get_allowed_concepts

from utils.plots import GT_COLS

from utils.constants import PIPE_MAX_NEW_TOKENS, MODEL_TEMPERATURE, BATCH_SIZE, PIPE_RETURN_FULL_TEXT, CONCEPT_TO_CHAPTER_MAPPING

In [8]:
data = "./data/final_dataset.csv"

df = pd.read_csv(data, sep=";")

In [9]:
demo_labels = [
    ["no", "no", "yes"], # All wrong
    ["yes", "yes", "no"], # All correct 
]
labels = demo_labels[0]

label_cols = df[GT_COLS]

In [42]:
def sample_dataset(df, seed, num_random_demos, type_of_demonstrations):
    if num_random_demos is None or num_random_demos <= 0:
        return None

    labels_pos = ["yes", "yes", "no"]
    labels_neg = ["no", "no", "yes"]

    # Get ground-truth evaluations
    label_cols = df[GT_COLS]

    # Only consider rows where all labels match
    match_pos = (label_cols == labels_pos).all(axis=1)
    match_neg = (label_cols == labels_neg).all(axis=1)

    match type_of_demonstrations:
        case -1:
            demos = df[match_neg].sample(num_random_demos, random_state=seed)
        case 0:
            half = num_random_demos // 2
            demos = pd.concat(
                [
                    df[match_neg].sample(half, random_state=seed),
                    df[match_pos].sample(num_random_demos - half, random_state=seed),
                ],
                #ignore_index=True,
            )
        case 1:
            demos = df[match_pos].sample(num_random_demos, random_state=seed)
        case _:
            raise ("Unknown type of demonstration!")

    return demos

In [46]:
sample_dataset(df, 0, 6, 0)

,title,problemDescription,exampleSolution,theme,topic,concept,The exercise description matched the selected theme (Yes/No),The exercise description matched the selected topic (Yes/No),Included concepts that were too advanced (Yes/No)
370,George RR Martin's Books Ratings,"Dua Lipa, the renowned pop artist, has a uniqu...","{""code"": ""import 'dart:io';main() { print('Wh...",literature,George RR Martin,conditional statements,no,no,yes
390,Beethoven's Symphony,Write a program that asks the user for their f...,"{""code"": ""import 'dart:io';\main() {\ print('...",classical music,Ludwig van Beethoven,program output,no,no,yes
351,Treasure Hunt Clues,"In a football tournament, there are different ...","{""code"": ""import 'dart:io';main() { print('Wh...",party games,Treasure Hunt,logical operators,no,no,yes
167,Favorite Adichie Book,Write a program that asks the user for their f...,"{'code': ""import 'dart:io';main() { print('Wh...",literature,Chimamanda Ngozi Adichie,program output,yes,yes,no
89,Limbo Game!,Write a program that asks the user for their n...,"{'code': ""import 'dart:io'; main() { print(...",party games,Limbo,program output,yes,yes,no
179,Music Trivia: Johann Sebastian Bach,Write a program that asks the user for their f...,"{'code': ""import 'dart:io';\nmain() {\nprint('...",classical music,Johann Sebastian Bach,program output,yes,yes,no
